In [1]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback  # 지금은 안 써도 됨 (원하면 제거 가능)
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")



# =========================
# RETRAIN WITH BEST PARAMS (CNN-LSTM)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params_cnnlstm = {
    "conv_filters": 48,
    "kernel_size": 4,
    "pool_size": 4,
    "lstm_units": 256,
    "dense_units": 16,
    "dropout_rate": 0.1156,
    "l2": 3.23e-05,       # 3.23E-05
    "lr": 5.71e-04,       # 5.71E-04
    "batch_size": 128
}

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))

# CNN 블록
x = layers.Conv1D(
    filters=best_params_cnnlstm["conv_filters"],
    kernel_size=best_params_cnnlstm["kernel_size"],
    activation="relu",
    padding="same",
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(inp)

x = layers.MaxPooling1D(pool_size=best_params_cnnlstm["pool_size"])(x)

# LSTM 블록
x = layers.LSTM(
    best_params_cnnlstm["lstm_units"],
    return_sequences=False,
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)

# Dense 블록
x = layers.Dense(
    best_params_cnnlstm["dense_units"],
    activation='relu',
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)
x = layers.Dropout(best_params_cnnlstm["dropout_rate"])(x)

out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params_cnnlstm["lr"])
best_model.compile(optimizer=opt, loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name='auc')])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params_cnnlstm["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)


# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte

n_features_used = Xte.shape[1]

# AIC/BIC 계산 포함 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params_cnnlstm
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

2025-12-17 06:21:47.794606: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 06:21:47.810063: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765920107.829071   90797 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765920107.835045   90797 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765920107.850107   90797 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)


[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


...

Epoch 38/50


547/547 - 6s - 10ms/step - auc: 0.9631 - loss: 0.1256 - val_auc: 0.9579 - val_loss: 0.1344 - learning_rate: 8.9219e-06


Epoch 39/50


547/547 - 6s - 10ms/step - auc: 0.9628 - loss: 0.1255 - val_auc: 0.9579 - val_loss: 0.1345 - learning_rate: 4.4609e-06


{
  "metrics": {
    "accuracy": 0.9467999999999368,
    "precision": 0.8499043977047324,
    "recall": 0.5810457516336072,
    "specificity": 0.9883444691907209,
    "f1": 0.6902173908214628,
    "roc_auc": 0.9652639853267603,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3846.6818985511227,
    "bic": 4158.929923234581,
    "log_likelihood": -1882.3409492755613,
    "n_features": 41,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "conv_filters": 48,
      "kernel_size": 4,
      "pool_size": 4,
      "lstm_units": 256,
      "dense_units": 16,
      "dropout_rate": 0.1156,
      "l2": 3.23e-05,
      "lr": 0.000571,
      "batch_size": 128
    }
  }
}
confusion_matrix [tn fp fn tp]: 13313 157 641 889


In [2]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback  # 지금은 안 써도 됨 (원하면 제거 가능)
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v1)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")



# =========================
# RETRAIN WITH BEST PARAMS (CNN-LSTM)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params_cnnlstm = {
    "conv_filters": 48,
    "kernel_size": 4,
    "pool_size": 4,
    "lstm_units": 256,
    "dense_units": 16,
    "dropout_rate": 0.1156,
    "l2": 3.23e-05,       # 3.23E-05
    "lr": 5.71e-04,       # 5.71E-04
    "batch_size": 128
}

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))

# CNN 블록
x = layers.Conv1D(
    filters=best_params_cnnlstm["conv_filters"],
    kernel_size=best_params_cnnlstm["kernel_size"],
    activation="relu",
    padding="same",
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(inp)

x = layers.MaxPooling1D(pool_size=best_params_cnnlstm["pool_size"])(x)

# LSTM 블록
x = layers.LSTM(
    best_params_cnnlstm["lstm_units"],
    return_sequences=False,
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)

# Dense 블록
x = layers.Dense(
    best_params_cnnlstm["dense_units"],
    activation='relu',
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)
x = layers.Dropout(best_params_cnnlstm["dropout_rate"])(x)

out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params_cnnlstm["lr"])
best_model.compile(optimizer=opt, loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name='auc')])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params_cnnlstm["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)


# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte

n_features_used = Xte.shape[1]

# AIC/BIC 계산 포함 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params_cnnlstm
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)


...

Epoch 43/50


547/547 - 6s - 10ms/step - auc: 0.9695 - loss: 0.1155 - val_auc: 0.9647 - val_loss: 0.1231 - learning_rate: 4.4609e-06


Epoch 44/50


547/547 - 6s - 11ms/step - auc: 0.9695 - loss: 0.1150 - val_auc: 0.9647 - val_loss: 0.1230 - learning_rate: 2.2305e-06


{
  "metrics": {
    "accuracy": 0.9520666666666031,
    "precision": 0.8560140474092572,
    "recall": 0.6372549019603678,
    "specificity": 0.9878247958425398,
    "f1": 0.7306107151340122,
    "roc_auc": 0.9699177062557766,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3534.0523327386563,
    "bic": 3869.1477738623676,
    "log_likelihood": -1723.0261663693282,
    "n_features": 44,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "conv_filters": 48,
      "kernel_size": 4,
      "pool_size": 4,
      "lstm_units": 256,
      "dense_units": 16,
      "dropout_rate": 0.1156,
      "l2": 3.23e-05,
      "lr": 0.000571,
      "batch_size": 128
    }
  }
}
confusion_matrix [tn fp fn tp]: 13306 164 555 975


In [3]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback  # 지금은 안 써도 됨 (원하면 제거 가능)
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v2)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")



# =========================
# RETRAIN WITH BEST PARAMS (CNN-LSTM)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params_cnnlstm = {
    "conv_filters": 48,
    "kernel_size": 4,
    "pool_size": 4,
    "lstm_units": 256,
    "dense_units": 16,
    "dropout_rate": 0.1156,
    "l2": 3.23e-05,       # 3.23E-05
    "lr": 5.71e-04,       # 5.71E-04
    "batch_size": 128
}

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))

# CNN 블록
x = layers.Conv1D(
    filters=best_params_cnnlstm["conv_filters"],
    kernel_size=best_params_cnnlstm["kernel_size"],
    activation="relu",
    padding="same",
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(inp)

x = layers.MaxPooling1D(pool_size=best_params_cnnlstm["pool_size"])(x)

# LSTM 블록
x = layers.LSTM(
    best_params_cnnlstm["lstm_units"],
    return_sequences=False,
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)

# Dense 블록
x = layers.Dense(
    best_params_cnnlstm["dense_units"],
    activation='relu',
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)
x = layers.Dropout(best_params_cnnlstm["dropout_rate"])(x)

out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params_cnnlstm["lr"])
best_model.compile(optimizer=opt, loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name='auc')])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params_cnnlstm["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)


# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte

n_features_used = Xte.shape[1]

# AIC/BIC 계산 포함 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params_cnnlstm
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)


...

Epoch 46/50


547/547 - 6s - 10ms/step - auc: 0.9752 - loss: 0.1053 - val_auc: 0.9717 - val_loss: 0.1128 - learning_rate: 4.4609e-06


Epoch 47/50


547/547 - 6s - 11ms/step - auc: 0.9747 - loss: 0.1058 - val_auc: 0.9719 - val_loss: 0.1128 - learning_rate: 2.2305e-06


{
  "metrics": {
    "accuracy": 0.9543999999999363,
    "precision": 0.8439024390237041,
    "recall": 0.6784313725485762,
    "specificity": 0.9857461024498154,
    "f1": 0.7521739125488406,
    "roc_auc": 0.9740240961516684,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3336.074487533987,
    "bic": 3694.0173450979514,
    "log_likelihood": -1621.0372437669935,
    "n_features": 47,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "conv_filters": 48,
      "kernel_size": 4,
      "pool_size": 4,
      "lstm_units": 256,
      "dense_units": 16,
      "dropout_rate": 0.1156,
      "l2": 3.23e-05,
      "lr": 0.000571,
      "batch_size": 128
    }
  }
}
confusion_matrix [tn fp fn tp]: 13278 192 492 1038


In [4]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback  # 지금은 안 써도 됨 (원하면 제거 가능)
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v3)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")



# =========================
# RETRAIN WITH BEST PARAMS (CNN-LSTM)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params_cnnlstm = {
    "conv_filters": 48,
    "kernel_size": 4,
    "pool_size": 4,
    "lstm_units": 256,
    "dense_units": 16,
    "dropout_rate": 0.1156,
    "l2": 3.23e-05,       # 3.23E-05
    "lr": 5.71e-04,       # 5.71E-04
    "batch_size": 128
}

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))

# CNN 블록
x = layers.Conv1D(
    filters=best_params_cnnlstm["conv_filters"],
    kernel_size=best_params_cnnlstm["kernel_size"],
    activation="relu",
    padding="same",
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(inp)

x = layers.MaxPooling1D(pool_size=best_params_cnnlstm["pool_size"])(x)

# LSTM 블록
x = layers.LSTM(
    best_params_cnnlstm["lstm_units"],
    return_sequences=False,
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)

# Dense 블록
x = layers.Dense(
    best_params_cnnlstm["dense_units"],
    activation='relu',
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)
x = layers.Dropout(best_params_cnnlstm["dropout_rate"])(x)

out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params_cnnlstm["lr"])
best_model.compile(optimizer=opt, loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name='auc')])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params_cnnlstm["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)


# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte

n_features_used = Xte.shape[1]

# AIC/BIC 계산 포함 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params_cnnlstm
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)


...

Epoch 27/50


547/547 - 6s - 11ms/step - auc: 0.9620 - loss: 0.1278 - val_auc: 0.9580 - val_loss: 0.1349 - learning_rate: 1.7844e-05


Epoch 28/50


547/547 - 6s - 10ms/step - auc: 0.9630 - loss: 0.1268 - val_auc: 0.9584 - val_loss: 0.1351 - learning_rate: 8.9219e-06


{
  "metrics": {
    "accuracy": 0.9465999999999368,
    "precision": 0.8454976303309522,
    "recall": 0.5830065359473313,
    "specificity": 0.98789903489228,
    "f1": 0.6901353960347237,
    "roc_auc": 0.9635986044998077,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3898.988451270235,
    "bic": 4279.778725274453,
    "log_likelihood": -1899.4942256351176,
    "n_features": 50,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "conv_filters": 48,
      "kernel_size": 4,
      "pool_size": 4,
      "lstm_units": 256,
      "dense_units": 16,
      "dropout_rate": 0.1156,
      "l2": 3.23e-05,
      "lr": 0.000571,
      "batch_size": 128
    }
  }
}
confusion_matrix [tn fp fn tp]: 13307 163 638 892


In [5]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback  # 지금은 안 써도 됨 (원하면 제거 가능)
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v4)# + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")



# =========================
# RETRAIN WITH BEST PARAMS (CNN-LSTM)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params_cnnlstm = {
    "conv_filters": 48,
    "kernel_size": 4,
    "pool_size": 4,
    "lstm_units": 256,
    "dense_units": 16,
    "dropout_rate": 0.1156,
    "l2": 3.23e-05,       # 3.23E-05
    "lr": 5.71e-04,       # 5.71E-04
    "batch_size": 128
}

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))

# CNN 블록
x = layers.Conv1D(
    filters=best_params_cnnlstm["conv_filters"],
    kernel_size=best_params_cnnlstm["kernel_size"],
    activation="relu",
    padding="same",
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(inp)

x = layers.MaxPooling1D(pool_size=best_params_cnnlstm["pool_size"])(x)

# LSTM 블록
x = layers.LSTM(
    best_params_cnnlstm["lstm_units"],
    return_sequences=False,
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)

# Dense 블록
x = layers.Dense(
    best_params_cnnlstm["dense_units"],
    activation='relu',
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)
x = layers.Dropout(best_params_cnnlstm["dropout_rate"])(x)

out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params_cnnlstm["lr"])
best_model.compile(optimizer=opt, loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name='auc')])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params_cnnlstm["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)


# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte

n_features_used = Xte.shape[1]

# AIC/BIC 계산 포함 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params_cnnlstm
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)
[MERGE] Checking v4...


   -> Added 7 features from v4 (ex: ['tn_sent_local', 'feat_topic_novelty', 'topic_N7']...)

[FINAL DATA] Total Shape: (579551, 124)


...

Epoch 35/50


547/547 - 6s - 10ms/step - auc: 0.9617 - loss: 0.1283 - val_auc: 0.9560 - val_loss: 0.1376 - learning_rate: 2.2305e-06


Epoch 36/50


547/547 - 6s - 11ms/step - auc: 0.9620 - loss: 0.1282 - val_auc: 0.9559 - val_loss: 0.1376 - learning_rate: 1.1152e-06


{
  "metrics": {
    "accuracy": 0.9465333333332702,
    "precision": 0.8493282149703941,
    "recall": 0.5784313725486415,
    "specificity": 0.9883444691907209,
    "f1": 0.6881804038720525,
    "roc_auc": 0.9634608498187336,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3923.2098787523937,
    "bic": 4273.536930836273,
    "log_likelihood": -1915.6049393761969,
    "n_features": 46,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "conv_filters": 48,
      "kernel_size": 4,
      "pool_size": 4,
      "lstm_units": 256,
      "dense_units": 16,
      "dropout_rate": 0.1156,
      "l2": 3.23e-05,
      "lr": 0.000571,
      "batch_size": 128
    }
  }
}
confusion_matrix [tn fp fn tp]: 13313 157 645 885


In [6]:
import os, random, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Tensorflow 관련 제거하고 Scikit-Learn RandomForest 추가
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models, optimizers, callbacks

# Optuna
import optuna
from optuna.integration import TFKerasPruningCallback  # 지금은 안 써도 됨 (원하면 제거 가능)
from sklearn.model_selection import StratifiedKFold
import gc
import xgboost as xgb

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# =========================
# CONFIG
# =========================
SEED = 1
MODE = "STRICT_CAUSAL"
SPLIT_STRATEGY = "RANDOM_ROW"
POS_RATIO = 0.10
TARGET_TOTAL = 100_000

EPOCHS = 50           # 최종 재학습 epoch
VAL_FRAC = 0.15
TEST_FRAC = 0.15
TIME_HOLDOUT_TEST_WEEKS = 8
TOL = 0.0025

# ✅ Optuna(튜닝) 전용 설정
N_TRIALS = 10             # Optuna trial 수
EPOCHS_TUNE = 15         # 튜닝용 epoch (CV 안에서)
MAX_TUNE_SAMPLES = 30_000  # fold별 train에서 최대 이만큼만 사용

# 최종 평가용 threshold
OPERATING_THR = 0.5
DATA_DIR = "."
# 고정 시드
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED);
rng = np.random.RandomState(SEED)

# 1. Base 데이터 (v0) 로드
path_v0 = os.path.join(DATA_DIR, "featured_v0.parquet")
print(f"[LOAD] Base: {path_v0}")
final_data = pd.read_parquet(path_v0)
print(f"   -> Base Shape: {final_data.shape}")

# 2. v1 ~ v4 순회하며 새로운 컬럼만 병합
versions = ["v1", "v2", "v3", "v4"]

for ver in versions:
    path_ver = os.path.join(DATA_DIR, f"featured_{ver}.parquet")
    if os.path.exists(path_ver):
        print(f"[MERGE] Checking {ver}...")
        temp_df = pd.read_parquet(path_ver)
        
        # v0(현재 final_data)에 없는 컬럼만 추출 (즉, 추가된 A, B, C, D 부분)
        new_cols = [c for c in temp_df.columns if c not in final_data.columns]
        
        if new_cols:
            # 인덱스 기준 병합 (axis=1). 행 순서가 동일하다고 가정합니다.
            # 만약 행 순서가 다르다면 merge(on='id')를 써야 하지만, 
            # 보통 FE 파이프라인 결과물은 순서가 같습니다.
            final_data = pd.concat([final_data, temp_df[new_cols]], axis=1)
            print(f"   -> Added {len(new_cols)} features from {ver} (ex: {new_cols[:3]}...)")
        else:
            print(f"   -> No new features in {ver}")
            
        # 메모리 정리
        del temp_df
        gc.collect()
    else:
        print(f"[WARN] File not found: {path_ver}")

print(f"\n[FINAL DATA] Total Shape: {final_data.shape}")
print("========================================")

# =========================
# 수동 metric + AUC 계산 함수
# =========================
def compute_roc_auc_manual(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    # NaN 제거
    mask = ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # 확률 내림차순 정렬
    order = np.argsort(-y_prob)
    y_true_sorted = y_true[order]

    cum_pos = np.cumsum(y_true_sorted == 1)
    cum_neg = np.cumsum(y_true_sorted == 0)

    tpr = cum_pos / n_pos
    fpr = cum_neg / n_neg

    # 사다리꼴 적분
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0
    for i in range(len(y_true_sorted)):
        auc += (fpr[i] - prev_fpr) * (tpr[i] + prev_tpr) / 2.0
        prev_fpr = fpr[i]
        prev_tpr = tpr[i]

    return float(auc)


# =========================
# 수동 metric + AUC + AIC/BIC 계산 함수
# =========================
def compute_aic_bic_manual(y_true, y_prob, n_features):
    """
    Log-Likelihood를 기반으로 AIC, BIC 계산
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    
    # Log(0) 방지를 위한 clipping (epsilon)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Log Likelihood 계산
    # L = sum( y*log(p) + (1-y)*log(1-p) )
    log_likelihood = np.sum(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    
    n = len(y_true)
    k = n_features
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return float(aic), float(bic), float(log_likelihood)

def compute_classification_metrics(y_true, y_prob, thr=OPERATING_THR, n_features=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    n = len(y_true)
    eps = 1e-9

    accuracy    = (tp + tn) / (n + eps)
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)

    roc_auc = compute_roc_auc_manual(y_true, y_prob)

    metrics = {
        "accuracy":       float(accuracy),
        "precision":      float(precision),
        "recall":         float(recall),
        "specificity":    float(specificity),
        "f1":             float(f1),
        "roc_auc":        float(roc_auc),
        "n_eval":         int(n),
        "pos_rate_eval":  float(y_true.mean()),
    }

    # [추가됨] n_features가 들어오면 AIC/BIC 계산
    if n_features is not None:
        aic, bic, ll = compute_aic_bic_manual(y_true, y_prob, n_features)
        metrics["aic"] = aic
        metrics["bic"] = bic
        metrics["log_likelihood"] = ll
        metrics["n_features"] = int(n_features)

    cm = (tn, fp, fn, tp)
    return metrics, cm


# =========================
# SPLIT
# =========================
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")
def brief(name, df):
    if df is None or len(df) == 0:
        print(f"[{name}] n=0"); return
    pos = int(df["is_purchased"].sum())
    n = len(df); rate = pos / max(n, 1)
    print(f"[{name}] n={n:,}  pos={pos:,}  rate={rate:.6f}")

def check_ratio(df, target_ratio=0.10, tol=TOL, name="dataset"):
    if len(df) == 0: return
    r = float(df["is_purchased"].mean())
    ok = abs(r - target_ratio) <= tol
    print(f"[check {name}] pos_rate={r:.6f}  target={target_ratio:.2f}  diff={r-target_ratio:+.6f}  {'OK' if ok else 'WARN'}")

def stratified_fixed_sample(df, target_col="is_purchased", total_n=100_000, pos_ratio=0.10, seed=SEED):
    n_pos_k = int(round(total_n * pos_ratio))
    n_neg_k = total_n - n_pos_k
    pos = df[df[target_col] == 1]
    neg = df[df[target_col] == 0]
    pos_s = pos.sample(n=min(n_pos_k, len(pos)), random_state=seed, replace=False)
    neg_s = neg.sample(n=min(n_neg_k, len(neg)), random_state=seed, replace=False)
    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

def split_random_row_simple(df, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    rng_local = np.random.RandomState(seed)
    N = len(df)
    idx = np.arange(N); rng_local.shuffle(idx)
    n_test = int(round(N * test_frac))
    n_val  = int(round(N * val_frac))
    te_idx = idx[:n_test]
    va_idx = idx[n_test:n_test+n_val]
    tr_idx = idx[n_test+n_val:]
    te = df.iloc[te_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)
    tr = df.iloc[tr_idx].reset_index(drop=True)
    return tr, va, te

dataset = final_data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
brief("dataset(raw)", dataset)
if TARGET_TOTAL and len(dataset) > TARGET_TOTAL:
    dataset = stratified_fixed_sample(dataset, target_col="is_purchased",
                                      total_n=TARGET_TOTAL, pos_ratio=POS_RATIO, seed=SEED)
brief("dataset(after 100K sampling)", dataset)
check_ratio(dataset, POS_RATIO, name="dataset(100K)")

if SPLIT_STRATEGY == "RANDOM_ROW":
    train_df, val_df, test_df = split_random_row_simple(dataset)
elif SPLIT_STRATEGY == "TIME_HOLDOUT":
    train_df, val_df, test_df = split_time_holdout(dataset)
else:
    train_df, val_df, test_df = split_user_disjoint(dataset)
    
TARGET = "is_purchased"

# =======================================================
# 1. Base Features (v0) - 총 41개
# : 캠페인, 채널, 플랫폼, 기본 이력 통계 등 가장 기초적인 정보
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

# =======================================================
# 2. Add-on Features (v1) - 총 3개
# : 구매 주기 및 행동 위험도 (Hazard/Refraction)
# =======================================================
cols_v1 = [
    'days_since_last_purchase',
    'feat_rtb_hazard',
    'feat_postbuy_refrac'
]

# =======================================================
# 3. Add-on Features (v2) - 총 6개
# : 캘린더 효과(주말/월초) 및 시간대 이동(Shift)
# =======================================================
cols_v2 = [
    'cal_is_weekend',
    'cal_week_of_month',
    'feat_dow_shift',
    'feat_eoq_bump',
    'feat_hour_shift',
    'feat_payday_bump'
]

# =======================================================
# 4. Add-on Features (v3) - 총 9개
# : 유저 피로도(Fatigue) 및 최근 30일 반응률(Context)
# =======================================================
cols_v3 = [
    'ctx_tc_open_rate_30d',
    'feat_fatigue',
    'feat_last_any_hours',
    'feat_last_email_hours',
    'feat_last_mobile_push_hours',
    'u_cadence_std_30d',
    'u_click_rate_30d',
    'u_open_cnt_30d',
    'u_open_rate_30d'
]

# =======================================================
# 5. Add-on Features (v4) - 총 5개
# : 토픽 신선도 및 행동 경로 유사성 (Sequence/Path)
# =======================================================
cols_v4 = [
    'feat_like_last_success',
    'feat_path_align',
    'feat_topic_novelty',
    'topic_N7',
    'topic_t_since_hours'
]

def get_feat_cols(df):
    """
    데이터프레임에서 학습에 사용할 최종 수치형 변수 리스트를 반환합니다.
    Target과 제외 리스트(ID, Leakage, Collinear)를 필터링합니다.
    """
    # 1. 모든 수치형 컬럼 추출
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 2. 제외 대상 필터링 (Target + Drop List)
    final_cols = list(cols_v0+ cols_v1 + cols_v2 + cols_v3 + cols_v4)
    
    # 3. 정렬 (모델 재현성을 위해 이름순 정렬)
    return sorted(final_cols)

# -------------------------------------------------------
# 실행 및 결과 확인
# -------------------------------------------------------
# train_df는 이미 로드되어 있다고 가정
feat_cols = get_feat_cols(train_df)
n_feat = len(feat_cols)

def prepare_data_3d(df, feature_columns):
    X = df[feature_columns].fillna(0).astype(np.float32).to_numpy()
    Y = df[TARGET].fillna(0).astype(np.int32).to_numpy()
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)  # (N, F, 1)
    return X_3d, Y

Xtr, Ytr = prepare_data_3d(train_df, feat_cols)
Xva, Yva = prepare_data_3d(val_df, feat_cols)
Xte, Yte = prepare_data_3d(test_df, feat_cols)

print("=== Data Shapes ===")
print(f"Train: X={Xtr.shape}, Y={Ytr.shape}")
print(f"Val  : X={Xva.shape}, Y={Yva.shape}")
print(f"Test : X={Xte.shape}, Y={Yte.shape}")



# =========================
# RETRAIN WITH BEST PARAMS (CNN-LSTM)
# =========================
print("\n[Final] Retraining model with best hyperparameters...")
best_params_cnnlstm = {
    "conv_filters": 48,
    "kernel_size": 4,
    "pool_size": 4,
    "lstm_units": 256,
    "dense_units": 16,
    "dropout_rate": 0.1156,
    "l2": 3.23e-05,       # 3.23E-05
    "lr": 5.71e-04,       # 5.71E-04
    "batch_size": 128
}

tf.keras.backend.clear_session()
inp = layers.Input(shape=(n_feat, 1))

# CNN 블록
x = layers.Conv1D(
    filters=best_params_cnnlstm["conv_filters"],
    kernel_size=best_params_cnnlstm["kernel_size"],
    activation="relu",
    padding="same",
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(inp)

x = layers.MaxPooling1D(pool_size=best_params_cnnlstm["pool_size"])(x)

# LSTM 블록
x = layers.LSTM(
    best_params_cnnlstm["lstm_units"],
    return_sequences=False,
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)

# Dense 블록
x = layers.Dense(
    best_params_cnnlstm["dense_units"],
    activation='relu',
    kernel_regularizer=regularizers.l2(best_params_cnnlstm["l2"]),
)(x)
x = layers.Dropout(best_params_cnnlstm["dropout_rate"])(x)

out = layers.Dense(1, activation="sigmoid")(x)

best_model = models.Model(inp, out)
opt = optimizers.Adam(learning_rate=best_params_cnnlstm["lr"])
best_model.compile(optimizer=opt, loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name='auc')])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = best_model.fit(
    Xtr, Ytr,
    validation_data=(Xva, Yva),
    epochs=EPOCHS,
    batch_size=best_params_cnnlstm["batch_size"],
    verbose=2,
    callbacks=[es, rlr]
)


# =========================
# FINAL EVALUATION (수동 metric)
# =========================
y_prob = best_model.predict(Xte, verbose=0).ravel()
y_true = Yte

n_features_used = Xte.shape[1]

# AIC/BIC 계산 포함 평가
final_metrics, (tn, fp, fn, tp) = compute_classification_metrics(
    y_true, y_prob, thr=OPERATING_THR, n_features=n_features_used
)

final_metrics.update({
    "mode": MODE,
    "split": SPLIT_STRATEGY,
    "best_params": best_params_cnnlstm
})

print(json.dumps({"metrics": final_metrics}, indent=2))
print("confusion_matrix [tn fp fn tp]:", tn, fp, fn, tp)

[LOAD] Base: ./featured_v0.parquet
   -> Base Shape: (433520, 62)
[MERGE] Checking v1...


   -> Added 3 features from v1 (ex: ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']...)
[MERGE] Checking v2...


   -> Added 7 features from v2 (ex: ['feat_hour_shift', 'feat_dow_shift', 'feat_payday_bump']...)
[MERGE] Checking v3...


   -> Added 45 features from v3 (ex: ['is_hard_bounced', 'hard_bounced_at', 'is_soft_bounced']...)


[MERGE] Checking v4...


...

Epoch 42/50


547/547 - 6s - 11ms/step - auc: 0.9686 - loss: 0.1179 - val_auc: 0.9623 - val_loss: 0.1300 - learning_rate: 4.4609e-06


Epoch 43/50


547/547 - 6s - 11ms/step - auc: 0.9688 - loss: 0.1177 - val_auc: 0.9625 - val_loss: 0.1300 - learning_rate: 2.2305e-06


{
  "metrics": {
    "accuracy": 0.9485999999999367,
    "precision": 0.8202531645562698,
    "recall": 0.6352941176466436,
    "specificity": 0.9841870824052721,
    "f1": 0.7160220989550599,
    "roc_auc": 0.9663286121179459,
    "n_eval": 15000,
    "pos_rate_eval": 0.102,
    "aic": 3777.9358287364994,
    "bic": 4265.347379461898,
    "log_likelihood": -1824.9679143682497,
    "n_features": 64,
    "mode": "STRICT_CAUSAL",
    "split": "RANDOM_ROW",
    "best_params": {
      "conv_filters": 48,
      "kernel_size": 4,
      "pool_size": 4,
      "lstm_units": 256,
      "dense_units": 16,
      "dropout_rate": 0.1156,
      "l2": 3.23e-05,
      "lr": 0.000571,
      "batch_size": 128
    }
  }
}
confusion_matrix [tn fp fn tp]: 13257 213 558 972
